# Finch — Phase D1/D2: Kaggle -> Clean & Split-Adjusted -> GCS (`finch-514709`)
Run in **Google Colab** (High-RAM runtime recommended).

In [1]:
# 1. Authenticate to GCP & set project
from google.colab import auth
auth.authenticate_user()
!gcloud config set project finch-514709

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



In [2]:
# 2. Clone your Finch repo & install dependencies
REPO_URL = "https://github.com/gautham9566/finch.git"
!git clone {REPO_URL} /content/finch || (cd /content/finch && git pull)
%cd /content/finch
!pip install -q -e .

Cloning into '/content/finch'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 33 (delta 6), reused 32 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 8.69 KiB | 1.24 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/finch
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for finch (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.


In [ ]:
# 3. Upload kaggle.json (never commit this to GitHub)
from google.colab import files
import json, os
os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()
    !mv kaggle.json /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json
cfg = json.load(open('/root/.kaggle/kaggle.json'))
if 'KAGGLE_API_TOKEN' in cfg:
    os.environ['KAGGLE_API_TOKEN'] = cfg['KAGGLE_API_TOKEN']
    open('/root/.kaggle/access_token', 'w').write(cfg['KAGGLE_API_TOKEN'])
    os.chmod('/root/.kaggle/access_token', 0o600)


In [ ]:
# 4. Download Kaggle 1-min dataset, validate, and push raw/clean to GCS
!python src/data/etl_kaggle_to_gcs.py
!python src/data/validate_1min.py /tmp/finch_kaggle/equities_1min.parquet

In [ ]:
# 5. Fetch splits & FII/DII, adjust 1-min prices, and upload adjusted Parquet to GCS
import duckdb
from src.data.nse_eod_fetcher import fetch_splits_yfinance, fetch_fii_dii
from src.data.corp_actions import adjust_1min_parquet

symbols = [r[0] for r in duckdb.sql("SELECT DISTINCT symbol FROM read_parquet('/tmp/finch_kaggle/equities_1min.parquet')").fetchall()]
fetch_splits_yfinance(symbols, '/tmp/finch_splits.parquet')
fetch_fii_dii('/tmp/finch_fii_dii.parquet')
adjust_1min_parquet('/tmp/finch_clean_equities/**/*.parquet', '/tmp/finch_splits.parquet', '/tmp/finch_adjusted_equities')
!gsutil -m cp -r /tmp/finch_adjusted_equities gs://finch-514709-lake/clean/equities_1min_adj/
!gsutil cp /tmp/finch_splits.parquet /tmp/finch_fii_dii.parquet gs://finch-514709-lake/raw/nse_meta/